In [25]:
import numpy as np
from tqdm import tqdm

def generate_data():
    """
    生成 100 组节点数据：
      - 2 节点：10 组
      - 3 节点：60 组
      - 4 节点：30 组
    每组随机生成优先级 p_i ∈ {1, 2, 3}
    返回一个列表 data_sets，长度为 100，每个元素为 (n, p_array)
    """
    data_sets = []

    # 2 节点，10 组
    for _ in range(100):
        n = 2
        p = np.random.randint(1, 4, size=n)  # p ∈ {1,2,3}
        data_sets.append((n, p))

    # 3 节点，60 组
    for _ in range(600):
        n = 3
        p = np.random.randint(1, 4, size=n)
        data_sets.append((n, p))

    # 4 节点，30 组
    for _ in range(300):
        n = 4
        p = np.random.randint(1, 4, size=n)
        data_sets.append((n, p))

    return data_sets

def compute_sending_rates(n, p_array, w1, w2, w3, w4, xi_out, sigma):
    """
    根据给定的 n, p_array，以及参数 w1, w2, w3, w4, xi_out, sigma，
    计算该组节点的所有 sending_rate(i)，并返回对应的数组。

    公式:
    sending_rate(i) = -1 + w1 * xi_out / (w2 * n * sigma
                                         + w3 * (p_i - avg_p) * xi_out
                                         + w4 * xi_out)
    """
    avg_p = np.mean(p_array)
    
    # 不随 p_i 变化的分母部分
    denominator_base = w2 * n * sigma + w4 * xi_out

    sending_rates = []
    for p_i in p_array:
        denominator = denominator_base + w3 * (p_i - avg_p) * xi_out
        rate = -1 + w1 * xi_out / denominator
        sending_rates.append(rate)
    
    return np.array(sending_rates)

def compute_metrics_for_params(w2, w3, w4, data_sets, w1, xi_out, sigma, m):
    """
    功能：给定 w2, w3, w4，计算并返回以下四个量：
      1) 100 组节点数据的 平均 Throughput
      2) 混合所有节点数据时的 WFI
      3) 100 组中最大的 (sum(sending_rate)*sigma / xi_out)
      4) 100 组 Throughput 的方差

    参数:
      - w2, w3, w4: 待搜索的参数
      - data_sets:  之前生成的 100 组节点数据, 格式 [(n, p_array), ...]
      - w1, xi_out, sigma, m: 其他固定参数，m 为 100 组节点的总数 (320)

    返回:
      (avg_throughput, wfi, max_ratio, var_throughput)
    """
    throughput_list = []
    srp_all = []            # sending_rate(i) * p_i
    srp_square_all = []     # (sending_rate(i)*p_i)^2
    ratio_list = []         # (sum(sending_rate)*sigma / xi_out) 各组的值

    for (n, p_array) in data_sets:
        # 计算该组下的 sending_rate
        sr = compute_sending_rates(n, p_array, w1, w2, w3, w4, xi_out, sigma)
        sum_sr = np.sum(sr)

        # throughput
        throughput_list.append(sum_sr * sigma)

        # WFI 需要 sending_rate(i)*p_i
        srp = sr * p_array
        srp_all.extend(srp)
        srp_square_all.extend((srp)**2)  # (sending_rate(i)*p_i)^2

        # ratio
        ratio_list.append((sum_sr * sigma) / xi_out)

    # (1) 平均 Throughput
    avg_throughput = np.mean(throughput_list)

    # (2) WFI
    srp_all = np.array(srp_all)
    srp_square_all = np.array(srp_square_all)
    numerator = (np.sum(srp_all))**2
    denominator = m * np.sum(srp_square_all)
    wfi = numerator / denominator if denominator != 0 else 0.0

    # (3) max_ratio
    max_ratio = np.max(ratio_list)

    # (4) 100 组 Throughput 的方差
    var_throughput = np.var(throughput_list)

    return (avg_throughput, wfi, max_ratio, var_throughput)

def compute_matrix_for_w2(w2, w3_values, w4_values, data_sets, w1, xi_out, sigma, m):
    """
    在固定 w2 的情况下，以 w3, w4 为横纵坐标，构造一个矩阵，
    每个元素为 (avg_throughput, wfi, max_ratio, var_throughput)

    参数:
      - w2: 固定参数
      - w3_values, w4_values: w3, w4 循环取值列表
      - data_sets:  100 组节点数据
      - w1, xi_out, sigma, m: 其余固定参数
    
    返回:
      一个二维 list，形状 (len(w3_values), len(w4_values))，
      每个元素为 (avg_throughput, wfi, max_ratio, var_throughput)
    """
    matrix_result = []

    for w3 in w3_values:
        row_result = []
        for w4 in w4_values:
            metrics = compute_metrics_for_params(
                w2, w3, w4, data_sets, w1, xi_out, sigma, m
            )
            row_result.append(metrics)
        matrix_result.append(row_result)

    return matrix_result

In [31]:
# 固定参数
xi_out = 12.8
sigma = 0.95
w1 = 15
m = 2*100 + 3*600 + 4*300  # 总节点数 3200

# 生成 100 组数据
data_sets = generate_data()

# 示例：搜索范围
w2_values = np.arange(6, 15.1, 0.5)
w3_values = np.arange(0.8, 1.3, 0.1)
w4_values = np.arange(0.2, 2.1, 0.2)

# ============ 1) 演示：查看某一个 (w2, w3, w4) 的四项指标 ============
example_w2 = 10.0
example_w3 = 1.0
example_w4 = 1.0
example_metrics = compute_metrics_for_params(
    example_w2, example_w3, example_w4, 
    data_sets, w1, xi_out, sigma, m
)
print(f"[单次调用] 对参数 (w2={example_w2}, w3={example_w3}, w4={example_w4}) 的四项指标：")
print("  平均Throughput =", example_metrics[0])
print("  WFI           =", example_metrics[1])
print("  最大 ratio    =", example_metrics[2])
print("  Throughput方差 =", example_metrics[3])
print()

[单次调用] 对参数 (w2=10.0, w3=1.0, w4=1.0) 的四项指标：
  平均Throughput = 10.932986039599262
  WFI           = 0.9011422702781751
  最大 ratio    = 0.9372831168946928
  Throughput方差 = 0.282191166802878



In [29]:
fixed_w2 = 10
matrix_result = compute_matrix_for_w2(fixed_w2, w3_values, w4_values,
                                        data_sets, w1, xi_out, sigma, m)

# matrix_result 是一个二维 list，[len(w3_values)][len(w4_values)] 的大小
# 这里演示只查看前 2 行、前 2 列
print(f"[矩阵] 固定 w2={fixed_w2}，行列分别对应 w3, w4:")
for i in range(5):
    for j in range(9):
        # (avg_throughput, wfi, max_ratio)
        print(f"w3={w3_values[i]}, w4={w4_values[j]} => {matrix_result[i][j]}")
    print()

[矩阵] 固定 w2=10，行列分别对应 w3, w4:
w3=0.8, w4=0.2 => (15.531116337457991, 0.88167848843674, 1.5585096298086536, 0.9146106166298953)
w3=0.8, w4=0.4 => (14.01735414089962, 0.8865025511752026, 1.2929459865955655, 0.41522948189005454)
w3=0.8, w4=0.6000000000000001 => (12.749559719166754, 0.8895257184371629, 1.1043204853065614, 0.22941884107352137)
w3=0.8, w4=0.8 => (11.668387976684956, 0.8914682695919818, 0.9708317820278629, 0.16535014740764264)
w3=0.8, w4=1.0 => (10.733235505240412, 0.8927253270936064, 0.8849638765763294, 0.14899493521910248)
w3=0.8, w4=1.2 => (9.91503774516182, 0.8935311072039668, 0.8121844042076521, 0.1499704576862004)
w3=0.8, w4=1.4000000000000001 => (9.192276793272725, 0.8940322371818974, 0.7578043105711991, 0.15558258897600646)
w3=0.8, w4=1.6 => (8.548595178837061, 0.8943243181561827, 0.708647445977331, 0.16070105844182142)
w3=0.8, w4=1.8 => (7.971288349828357, 0.8944716711319906, 0.6639802718788937, 0.16351067810228864)

w3=1.0, w4=0.2 => (16.16516134693846, 0.88158895999

In [27]:
def compute_matrix_with_constraint(w3_values, w4_values, w2_values, data_sets, w1, xi_out, sigma, m, throughput_threshold=1.2):
    """
    功能3：生成关于 w3 和 w4 的矩阵，其中每个位置点对应的 w2 是在 [6, 12] 中取值，
    使得对应的 max_throughput <= throughput_threshold。
    如果存在多个 w2 满足条件，选择最小的 w2。
    如果不存在，则记录为 None。

    参数:
      - w3_values, w4_values: w3, w4 循环取值列表
      - w2_values: w2 循环取值列表
      - data_sets: 100 组节点数据
      - w1, xi_out, sigma, m: 其余固定参数
      - throughput_threshold: max_throughput 的阈值，默认为 1.2

    返回:
      一个二维 list，形状 (len(w3_values), len(w4_values))，
      每个元素为 (w2, avg_throughput, wfi, max_ratio, var_throughput) 或 None
    """
    matrix_result = []

    for w3 in tqdm(w3_values, desc="Loop over w3"):
        row_result = []
        for w4 in w4_values:
            selected_w2 = None
            selected_metrics = None
            # 遍历 w2_values，选择最小的 w2 满足 max_throughput <= threshold
            for w2 in w2_values:
                metrics = compute_metrics_for_params(
                    w2, w3, w4, data_sets, w1, xi_out, sigma, m
                )
                max_throughput = metrics[2]  # 根据 ratio: (sum(sr)*sigma)/xi_out = max_ratio
                if max_throughput <= throughput_threshold:
                    selected_w2 = w2
                    selected_metrics = metrics
                    break  # 找到最小的 w2 满足条件，退出循环
            if selected_w2 is not None:
                row_result.append((selected_w2, *selected_metrics))
            else:
                row_result.append(None)  # 或者其他标记，如 (None, None, None, None, None)
        matrix_result.append(row_result)

    return matrix_result

In [32]:
throughput_threshold = 1.02
print(f"开始生成 (w3, w4) 矩阵，选择 w2 使得 max_throughput <= {throughput_threshold} ...")
matrix_with_constraint = compute_matrix_with_constraint(
    w3_values, w4_values, w2_values, data_sets, w1, xi_out, sigma, m, throughput_threshold
)

# matrix_with_constraint 是一个二维 list，
# 大小为 [len(w3_values)][len(w4_values)]，
# 每个元素为 (w2, avg_throughput, wfi, max_ratio, var_throughput) 或 None
for i in range(5):
    for j in range(9):
        # (avg_throughput, wfi, max_ratio)
        print(f"w3={w3_values[i]}, w4={w4_values[j]} => {matrix_with_constraint[i][j]}")
    print()

开始生成 (w3, w4) 矩阵，选择 w2 使得 max_throughput <= 1.02 ...


Loop over w3: 100%|██████████| 5/5 [00:04<00:00,  1.24it/s]

w3=0.8, w4=0.2 => (13.5, 10.67522159900543, 0.8687644190110724, 1.0151619972114339, 0.34781991939343004)
w3=0.8, w4=0.4 => (12.5, 10.805023353780525, 0.8763444605778962, 0.9808147720712997, 0.20337870795386584)
w3=0.8, w4=0.6000000000000001 => (11.0, 11.488031647252502, 0.8857495706762607, 0.9956315107348909, 0.14840823549900603)
w3=0.8, w4=0.8 => (10.0, 11.646369311521608, 0.8923638843873769, 0.9708317820278629, 0.15997802434196992)
w3=0.8, w4=1.0 => (9.0, 11.814386517332986, 0.8984588424919222, 0.9814810650375168, 0.2593991587691961)
w3=0.8, w4=1.2 => (8.0, 11.992609994575817, 0.9040541892132733, 1.0144184362934363, 0.4534394032802121)
w3=0.8, w4=1.4000000000000001 => (7.5, 11.579205869095906, 0.9065766200206541, 0.9938533795167005, 0.5797634329641757)
w3=0.8, w4=1.6 => (7.0, 11.19185500621208, 0.9086970037951236, 0.9739492883543102, 0.7157501849553556)
w3=0.8, w4=1.8 => (6.0, 11.368957077912668, 0.9129119681648801, 1.0118893655131511, 1.0750647507518074)

w3=0.9, w4=0.2 => (14.0, 10

: 